# Coach Trip Table Build-Up From TransXChange

This notebook mirrors the teaching style of `03_bus_mobility.ipynb`, but for
the coach XML files stored in `Data/EV_behavior/Coach_Data/TxC-2.4`.

Goal:

1. inspect the coach inventory tables;
2. open one raw TransXChange XML file;
3. parse `StopPoints`, `JourneyPatterns`, and `VehicleJourneys`;
4. expand each `VehicleJourney` into timing rows;
5. collapse that into a one-row-per-trip table.

Important distinction:

- in the coach XML, `VehicleJourney` is the closest thing to a "trip";
- it is **not** yet a vehicle block like the `block_id` workflow in GTFS bus data.


In [ ]:
from pathlib import Path

import pandas as pd

from mobility.coach.txc_parser import (
    load_txc_components,
    build_journey_pattern_link_table,
    expand_vehicle_journeys_to_timing_rows,
    build_vehicle_journey_stop_times,
    build_trip_table_from_xml,
    build_trip_table_from_inventory,
)

COACH = Path('../../Data/EV_behavior/Coach_Data/TxC-2.4')
INCLUDED = COACH / 'IncludedServices17APR26.csv'
INVENTORY = COACH / 'TxCInventory17APR26.csv'
SAMPLE_XML = COACH / 'NATX' / 'NATX-National_Express-400-London-Wolverhampton.xml'

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

print(f'Coach root: {COACH.resolve()}')
print(f'Inventory exists: {INVENTORY.exists()}')
print(f'Sample XML exists: {SAMPLE_XML.exists()}')


## 1. Inventory tables

The coach folder already contains two useful CSV summaries:

- `IncludedServices17APR26.csv`: a compact service-direction list
- `TxCInventory17APR26.csv`: one row per XML file, with counts of stops,
  route sections, journey patterns, and vehicle journeys


In [ ]:
included = pd.read_csv(INCLUDED)
inventory = pd.read_csv(INVENTORY)

print('Included services:')
display(included.head(10))

print('Inventory:')
display(inventory.head(10))


In [ ]:
inventory.groupby('NationalOperatorCode').agg(
    xml_files=('FilePath', 'count'),
    vehicle_journeys=('VehicleJourneys', 'sum'),
    total_stop_points=('TotalStopPoints', 'sum'),
).sort_values('vehicle_journeys', ascending=False)


## 2. Pick one sample XML

We use National Express line `400` as a readable example. The file contains:

- one service definition (`ServiceCode`, `LineName`)
- several route variants / journey patterns
- many vehicle journeys with different departure times and operating dates


In [ ]:
components = load_txc_components(SAMPLE_XML)
components['metadata']


In [ ]:
for name in [
    'stop_points',
    'route_sections',
    'routes',
    'journey_pattern_sections',
    'journey_patterns',
    'vehicle_journeys',
    'vehicle_journey_timing_links',
]:
    df = components[name]
    print(f'{name:28s} shape={df.shape}')


## 3. Core XML layers

The important build-up is:

`StopPoints -> RouteSections -> Routes -> JourneyPatterns -> VehicleJourneys`

In practical terms:

- `JourneyPattern` says *which stop sequence / path variant* is used;
- `VehicleJourney` says *when that pattern actually departs* and on which dates.


In [ ]:
display(components['stop_points'].head(10))
display(components['routes'].head(10))
display(components['journey_patterns'].head(10))
display(components['vehicle_journeys'].head(10))


## 4. Flatten JourneyPattern to ordered timing links

`JourneyPatternSectionRefs` point into `JourneyPatternSections`, which contain
the ordered stop-to-stop timing links.

This is the first major step toward a trip table: turn the nested XML
references into an ordered tabular chain.


In [ ]:
jp_links = build_journey_pattern_link_table(components)
print(jp_links.shape)
jp_links.head(20)


In [ ]:
jp_links.groupby(['journey_pattern_id', 'direction', 'pattern_description']).agg(
    n_links=('journey_pattern_timing_link_id', 'count'),
    start_stop=('from_stop_ref', 'first'),
    end_stop=('to_stop_ref', 'last'),
).reset_index().head(15)


## 5. Expand VehicleJourney into detailed timing rows

Now each `VehicleJourney` inherits:

- its departure time;
- the ordered timing links of its `JourneyPatternRef`;
- any runtime overrides defined directly on `VehicleJourneyTimingLink`.

Result: one row per trip-segment.


In [ ]:
timing_rows = expand_vehicle_journeys_to_timing_rows(components)
print(timing_rows.shape)
timing_rows.head(20)


In [ ]:
sample_vj = timing_rows['vehicle_journey_code'].iloc[0]
timing_rows[timing_rows['vehicle_journey_code'] == sample_vj][
    [
        'vehicle_journey_code',
        'journey_pattern_ref',
        'direction',
        'pattern_link_order',
        'from_stop_ref',
        'to_stop_ref',
        'segment_start_time',
        'segment_end_time',
        'runtime_s',
        'wait_s',
    ]
]


## 6. Stop-level view

For teaching, it is often easier to inspect stop-sequence rows before jumping
to the final trip table.


In [ ]:
stop_times = build_vehicle_journey_stop_times(timing_rows, components['stop_points'])
print(stop_times.shape)
stop_times.head(20)


In [ ]:
stop_times[stop_times['vehicle_journey_code'] == sample_vj]


## 7. Final trip table for one XML

Here we collapse each `VehicleJourney` to one row:

- start stop / end stop
- departure / arrival
- number of stops
- total runtime
- operating-profile summary


In [ ]:
trip_table = build_trip_table_from_xml(SAMPLE_XML)
print(trip_table.shape)
trip_table.head(20)


In [ ]:
trip_table.groupby(['direction', 'pattern_description']).agg(
    trips=('vehicle_journey_code', 'count'),
    first_dep=('departure_time', 'min'),
    last_dep=('departure_time', 'max'),
    mean_runtime_min=('runtime_min', 'mean'),
).reset_index().sort_values(['direction', 'first_dep'])


## 8. Batch conversion for a few files

The same parser can be used across multiple XML files. Below we run a small
sample so the process stays easy to inspect in one notebook.


In [ ]:
natx_sample = inventory[inventory['NationalOperatorCode'] == 'NATX'].head(3).copy()
natx_sample[['LineName', 'FilePath', 'VehicleJourneys']]


In [ ]:
batch_trip_table = build_trip_table_from_inventory(natx_sample, COACH)
print(batch_trip_table.shape)
batch_trip_table.head(20)


In [ ]:
batch_trip_table.groupby(['OperatorShortName', 'LineName']).agg(
    trips=('vehicle_journey_code', 'count'),
    mean_runtime_min=('runtime_min', 'mean'),
    min_dep=('departure_time', 'min'),
    max_dep=('departure_time', 'max'),
).reset_index()


## 9. What this notebook gives you

After this conversion, the next modelling step would usually be:

1. treat each `VehicleJourney` as a coach trip row;
2. infer chaining between trips to reconstruct vehicle duties / blocks;
3. attach stop coordinates or distances if you want energy modelling.

That is exactly the stage where the coach workflow starts to resemble the
`mobility.bus` workflow.
